In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Gemini API key not found in .env file")

print("Gemini API key loaded successfully!")

Gemini API key loaded successfully!


In [7]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": True,
    }
)

In [8]:
## Question 1 – Sequential Task Pipeline

from autogen_agentchat.agents import AssistantAgent

draft_writer = AssistantAgent(
    name="DraftWriter",
    model_client=model_client,
    system_message=(
        "You are an article writer. Write a short, informative article "
        "on the given topic."
    )
)

editor = AssistantAgent(
    name="EditorAgent",
    model_client=model_client,
    system_message=(
        "You are a professional editor. Improve grammar, structure, "
        "clarity, and tone while preserving the original meaning."
    )
)

plagiarism_checker = AssistantAgent(
    name="PlagiarismChecker",
    model_client=model_client,
    system_message=(
        "Review the article for potentially copied or overly familiar "
        "phrasing. Identify suspicious passages and suggest improvements. "
        "Do not claim to have verified plagiarism against external databases."
    )
)

publisher = AssistantAgent(
    name="PublisherAgent",
    model_client=model_client,
    system_message=(
        "You are a publishing editor. Prepare the reviewed article "
        "with a suitable title, headings, and final formatting."
    )
)

topic = "The Impact of Artificial Intelligence on Education"

result1 = await draft_writer.run(task=f"Write an article about: {topic}")
draft = result1.messages[-1].content
print("\n--- Draft Writer\n", draft)

result2 = await editor.run(task=f"Edit this draft:\n{draft}")
edited = result2.messages[-1].content
print("\n Editor", edited)

result3 = await plagiarism_checker.run(
    task=f"Review this article for originality:\n{edited}"
)
checked = result3.messages[-1].content
print("\n Plagiarism Checker -\n", checked)

result4 = await publisher.run(
    task=f"Finalize this article for publishing:\n{checked}"
)
final_article = result4.messages[-1].content
print("\n Publisher Output \n", final_article)


--- Draft Writer
 **The Impact of Artificial Intelligence on Education: A Paradigm Shift**

Artificial Intelligence (AI) is no longer a futuristic concept confined to science fiction; it is actively reshaping our present. While industries ranging from healthcare to finance have rapidly embraced AI, the education sector is currently undergoing its own profound transformation. From personalized learning to automated grading, AI is redefining how students learn and how educators teach. 

### Personalized Learning at Scale
One of the most significant impacts of AI in education is the shift toward personalized learning. Traditionally, classrooms have operated on a "one-size-fits-all" model, where a single teacher must cater to the diverse needs of thirty or more students. 

AI-powered educational platforms change this dynamic by analyzing a student’s strengths, weaknesses, and learning pace in real-time. Algorithms can identify if a student is struggling with a specific math concept or gra

In [9]:
## Question 2 – Task Decomposer (Planner + Workers)

from autogen_agentchat.agents import AssistantAgent

planner = AssistantAgent(
    name="PlannerAgent",
    model_client=model_client,
    system_message=(
        "You are a business planning expert. Break the given startup "
        "goal into market research, technical requirements, and financial "
        "planning tasks. Clearly describe each task."
    )
)

market_agent = AssistantAgent(
    name="MarketAgent",
    model_client=model_client,
    system_message=(
        "You are a market research analyst. Analyze the target customers, "
        "market opportunities, competitors, and business challenges."
    )
)

tech_agent = AssistantAgent(
    name="TechAgent",
    model_client=model_client,
    system_message=(
        "You are a technology consultant. Suggest the technology stack, "
        "product features, architecture, and development requirements."
    )
)

financial_agent = AssistantAgent(
    name="FinancialAgent",
    model_client=model_client,
    system_message=(
        "You are a financial analyst. Suggest startup costs, revenue "
        "models, operating expenses, and financial planning assumptions."
    )
)

goal = "Create a business plan for a startup in mental health tech."

plan_result = await planner.run(task=goal)
plan = plan_result.messages[-1].content
print("\n Planner Output \n", plan)


market_result = await market_agent.run(
    task=f"Based on this startup goal and plan, analyze the market:\n{goal}\n{plan}"
)
print("\n  Market Analysis \n", market_result.messages[-1].content)

tech_result = await tech_agent.run(
    task=f"Based on this startup goal and plan, provide technical planning:\n{goal}\n{plan}"
)
print("\n Technical Plan \n", tech_result.messages[-1].content)

finance_result = await financial_agent.run(
    task=f"Based on this startup goal and plan, provide financial planning:\n{goal}\n{plan}"
)
print("\n  Financial Plan \n", finance_result.messages[-1].content)


 Planner Output 
 Welcome to the business planning framework for your mental health tech startup. To turn your vision into a viable business, we must break down your overarching goal into three foundational pillars: **Market Research**, **Technical Requirements**, and **Financial Planning**. 

Here is the detailed task breakdown for each category.

---

### Phase 1: Market Research
*Objective: Validate the problem, understand the target audience, analyze the competition, and navigate the regulatory landscape.*

1. **Target Audience & Persona Definition**
   * *Description:* Define your primary users (e.g., individuals with mild-to-moderate anxiety, corporate employees, college students) and secondary buyers (e.g., HR departments, health insurance providers, or therapists). Conduct interviews and surveys to map their pain points, willingness to pay, and preferred digital channels.
2. **Competitive Landscape & Gap Analysis**
   * *Description:* Audit existing mental health apps (e.g., H

In [ ]:
## Question 3 – Role-Based Chat (Specialized Roles)

from autogen_agentchat.agents import AssistantAgent

# Doctor Agent
doctor = AssistantAgent(
    name="DoctorAgent",
    model_client=model_client,
    system_message=(
        "You provide general educational information about diabetes "
        "management. Never diagnose or recommend changing medication. "
        "Encourage users to consult their healthcare professional."
    )
)

# Dietitian Agent
dietitian = AssistantAgent(
    name="DietitianAgent",
    model_client=model_client,
    system_message=(
        "You are a dietitian assistant. Suggest general balanced meal "
        "ideas for diabetes management. Consider nutrition and portion "
        "awareness. Avoid prescribing an individualized medical diet."
    )
)

# Fitness Coach Agent
fitness_coach = AssistantAgent(
    name="FitnessCoachAgent",
    model_client=model_client,
    system_message=(
        "You are a fitness coach. Suggest general, safe physical activity "
        "ideas for people managing diabetes. Recommend medical guidance "
        "before starting a new exercise routine."
    )
)

query = "Help me manage diabetes naturally."

# Step 1: Medical insight
doctor_result = await doctor.run(task=query)
doctor_output = doctor_result.messages[-1].content
print("\n--- Doctor Advice ---\n", doctor_output)

# Step 2: Meal plan
diet_result = await dietitian.run(
    task=f"Suggest general meal ideas based on this query and context:\n{query}\n{doctor_output}"
)
diet_output = diet_result.messages[-1].content
print("\n--- Dietitian Suggestions ---\n", diet_output)

# Step 3: Workout suggestions
fitness_result = await fitness_coach.run(
    task=f"Suggest suitable general physical activities using this context:\n{query}\n{doctor_output}\n{diet_output}"
)
print("\n--- Fitness Coach Suggestions ---\n", fitness_result.messages[-1].content)

In [5]:
## Question 4 – Multi-Language Translator Agents

from autogen_agentchat.agents import AssistantAgent

english_to_spanish = AssistantAgent(
    name="EnglishToSpanish",
    model_client=model_client,
    system_message=(
        "Translate the given English text into Spanish. "
        "Return only the translation."
    )
)

spanish_to_french = AssistantAgent(
    name="SpanishToFrench",
    model_client=model_client,
    system_message=(
        "Translate the given Spanish text into French. "
        "Return only the translation."
    )
)

french_to_german = AssistantAgent(
    name="FrenchToGerman",
    model_client=model_client,
    system_message=(
        "Translate the given French text into German. "
        "Return only the translation."
    )
)

text = "Artificial Intelligence is transforming the world."

result1 = await english_to_spanish.run(task=text)
spanish_text = result1.messages[-1].content
print("\n--- Spanish Translation ---\n", spanish_text)

result2 = await spanish_to_french.run(task=spanish_text)
french_text = result2.messages[-1].content
print("\n--- French Translation ---\n", french_text)

result3 = await french_to_german.run(task=french_text)
german_text = result3.messages[-1].content
print("\n--- Final German Translation ---\n", german_text)


--- Spanish Translation ---
 La inteligencia artificial está transformando el mundo.

--- French Translation ---
 L'intelligence artificielle transforme le monde.

--- Final German Translation ---
 Künstliche Intelligenz verändert die Welt.
